#GOLD LAYER: BI READY TABLES

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType
from pyspark.sql import Row

In [0]:
catalog_name = "ecommerce"

##PRODUCTS

In [0]:
df_products = spark.table(f"{catalog_name}.silver.slv_products")
df_brands = spark.table(f"{catalog_name}.silver.slv_brands")
df_category = spark.table(f"{catalog_name}.silver.slv_category")

df_products.createOrReplaceTempView("v_products")
df_brands.createOrReplaceTempView("v_brands")
df_category.createOrReplaceTempView("v_category")

spark.sql(f"USE CATALOG {catalog_name}")

DataFrame[]

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW v_gld_dim_products AS

WITH brands_categories AS (
    SELECT
        b.brand_name,
        b.brand_code,
        c.category_name,
        c.category_code
    FROM v_brands b
    INNER JOIN v_category c
        ON b.category_code = c.category_code
)

SELECT
    p.product_id,
    p.sku,
    p.category_code,
    COALESCE(bc.category_name, 'Not Available') AS category_name,
    p.brand_code,
    COALESCE(bc.brand_name, 'Not Available') AS brand_name,
    p.color,
    p.size,
    p.material,
    p.weight_grams,
    p.length_cm,
    p.width_cm,
    p.height_cm,
    p.rating_count,
    p.file_name,
    p.ingest_timestamp
FROM v_products p
LEFT JOIN brands_categories bc
    ON p.brand_code = bc.brand_code;

In [0]:
%sql

CREATE OR REPLACE TABLE gold.gld_dim_products AS
SELECT *
FROM v_gld_dim_products;

num_affected_rows,num_inserted_rows


##CUSTOMERS

In [0]:
from pyspark.sql import Row

df_silver = spark.table(f"{catalog_name}.silver.slv_customers")



customer_id,phone,country_code,country,state,file_name,ingest_timestamp
CUST000000000005,618478772532.0,AU,Australia,WA,dbfs:/Volumes/ecommerce/raw/raw_landing/customers/customers.csv,2026-06-11T13:55:28.076Z
CUST000000000006,916441718520.0,IN,India,GJ,dbfs:/Volumes/ecommerce/raw/raw_landing/customers/customers.csv,2026-06-11T13:55:28.076Z
CUST000000000067,19984489004.0,CA,Canada,AB,dbfs:/Volumes/ecommerce/raw/raw_landing/customers/customers.csv,2026-06-11T13:55:28.076Z
CUST000000000075,919609908502.0,IN,India,UP,dbfs:/Volumes/ecommerce/raw/raw_landing/customers/customers.csv,2026-06-11T13:55:28.076Z
CUST000000000093,9717639144906.0,AE,United Arab Emirates,DU,dbfs:/Volumes/ecommerce/raw/raw_landing/customers/customers.csv,2026-06-11T13:55:28.076Z


In [0]:
# India states
india_region = {
    "MH": "West", "GJ": "West", "RJ": "West",
    "KA": "South", "TN": "South", "TS": "South", "AP": "South", "KL": "South",
    "UP": "North", "WB": "North", "DL": "North"
}

# Australia states
australia_region = {
    "VIC": "SouthEast", "WA": "West", "NSW": "East", "QLD": "NorthEast"
}

# United Kingdom states
uk_region = {
    "ENG": "England", "WLS": "Wales", "NIR": "Northern Ireland", "SCT": "Scotland"
}

# United States states
us_region = {
    "MA": "NorthEast", "FL": "South", "NJ": "NorthEast",
    "CA": "West", "NY": "NorthEast", "TX": "South"
}

# UAE states
uae_region = {
    "AUH": "Abu Dhabi", "DU": "Dubai", "SHJ": "Sharjah"
}

# Singapore states
singapore_region = {
    "SG": "Singapore"
}

# Canada states
canada_region = {
    "BC": "West", "AB": "West", "ON": "East",
    "QC": "East", "NS": "East", "IL": "Other"
}

country_state_map = {
    "India": india_region,
    "Australia": australia_region,
    "United Kingdom": uk_region,
    "United States": us_region,
    "United Arab Emirates": uae_region,
    "Singapore": singapore_region,
    "Canada": canada_region
}

# Create mapping dataframe
rows = []

for country, states in country_state_map.items():
    for state_code, region in states.items():
        rows.append(
            Row(
                country=country,
                state=state_code,
                region=region
            )
        )

df_region_mapping = spark.createDataFrame(rows)

# Join and enrich customer dimension
df_gold = (
    df_silver
    .join(
        df_region_mapping,
        on=["country", "state"],
        how="left"
    )
    .fillna({"region": "Other"})
)



country,state,customer_id,phone,country_code,file_name,ingest_timestamp,region
Australia,WA,CUST000000000005,618478772532.0,AU,dbfs:/Volumes/ecommerce/raw/raw_landing/customers/customers.csv,2026-06-11T13:55:28.076Z,West
India,GJ,CUST000000000006,916441718520.0,IN,dbfs:/Volumes/ecommerce/raw/raw_landing/customers/customers.csv,2026-06-11T13:55:28.076Z,West
Canada,AB,CUST000000000067,19984489004.0,CA,dbfs:/Volumes/ecommerce/raw/raw_landing/customers/customers.csv,2026-06-11T13:55:28.076Z,West
India,UP,CUST000000000075,919609908502.0,IN,dbfs:/Volumes/ecommerce/raw/raw_landing/customers/customers.csv,2026-06-11T13:55:28.076Z,North
United Arab Emirates,DU,CUST000000000093,9717639144906.0,AE,dbfs:/Volumes/ecommerce/raw/raw_landing/customers/customers.csv,2026-06-11T13:55:28.076Z,Dubai


In [0]:
desired_columns_order = ["customer_id", "phone", "country_code", "country", "state", "region", "file_name", "ingest_timestamp"]

df_gold = df_gold.select(desired_columns_order)

customer_id,phone,country_code,country,state,region,file_name,ingest_timestamp
CUST000000000005,618478772532.0,AU,Australia,WA,West,dbfs:/Volumes/ecommerce/raw/raw_landing/customers/customers.csv,2026-06-11T13:55:28.076Z
CUST000000000006,916441718520.0,IN,India,GJ,West,dbfs:/Volumes/ecommerce/raw/raw_landing/customers/customers.csv,2026-06-11T13:55:28.076Z
CUST000000000067,19984489004.0,CA,Canada,AB,West,dbfs:/Volumes/ecommerce/raw/raw_landing/customers/customers.csv,2026-06-11T13:55:28.076Z
CUST000000000075,919609908502.0,IN,India,UP,North,dbfs:/Volumes/ecommerce/raw/raw_landing/customers/customers.csv,2026-06-11T13:55:28.076Z
CUST000000000093,9717639144906.0,AE,United Arab Emirates,DU,Dubai,dbfs:/Volumes/ecommerce/raw/raw_landing/customers/customers.csv,2026-06-11T13:55:28.076Z


In [0]:
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.gold.gld_dim_customers")

##DATE

In [0]:
from pyspark.sql import functions as F

df_silver = spark.table(f"{catalog_name}.silver.slv_calendar")


date,year,day_name,quarter,_ingested_at,_source_file,week
2024-01-06,2024,Saturday,Q1-2024,2026-06-11T13:55:45.637Z,dbfs:/Volumes/ecommerce/raw/raw_landing/date/date.csv,Week-1-2024
2024-01-20,2024,Saturday,Q1-2024,2026-06-11T13:55:45.637Z,dbfs:/Volumes/ecommerce/raw/raw_landing/date/date.csv,Week-3-2024
2024-02-22,2024,Thursday,Q1-2024,2026-06-11T13:55:45.637Z,dbfs:/Volumes/ecommerce/raw/raw_landing/date/date.csv,Week-8-2024
2024-03-11,2024,Monday,Q1-2024,2026-06-11T13:55:45.637Z,dbfs:/Volumes/ecommerce/raw/raw_landing/date/date.csv,Week-11-2024
2024-03-26,2024,Tuesday,Q1-2024,2026-06-11T13:55:45.637Z,dbfs:/Volumes/ecommerce/raw/raw_landing/date/date.csv,Week-13-2024


In [0]:
df_gold = (
    df_silver

    # Create surrogate date key
    .withColumn(
        "date_id",
        F.date_format(F.col("date"), "yyyyMMdd").cast("int")
    )

    # Month name
    .withColumn(
        "month_name",
        F.date_format(F.col("date"), "MMMM")
    )

    # Weekend flag
    .withColumn(
        "is_weekend",
        F.when(
            F.col("day_name").isin("Saturday", "Sunday"),
            1
        ).otherwise(0)
    )
)

desired_columns_order = [
    "date_id",
    "date",
    "year",
    "month_name",
    "day_name",
    "is_weekend",
    "quarter",
    "week",
    "_ingested_at",
    "_source_file"
]

df_gold = df_gold.select(*desired_columns_order)



date_id,date,year,month_name,day_name,is_weekend,quarter,week,_ingested_at,_source_file
20240106,2024-01-06,2024,January,Saturday,1,Q1-2024,Week-1-2024,2026-06-11T13:55:45.637Z,dbfs:/Volumes/ecommerce/raw/raw_landing/date/date.csv
20240120,2024-01-20,2024,January,Saturday,1,Q1-2024,Week-3-2024,2026-06-11T13:55:45.637Z,dbfs:/Volumes/ecommerce/raw/raw_landing/date/date.csv
20240222,2024-02-22,2024,February,Thursday,0,Q1-2024,Week-8-2024,2026-06-11T13:55:45.637Z,dbfs:/Volumes/ecommerce/raw/raw_landing/date/date.csv
20240311,2024-03-11,2024,March,Monday,0,Q1-2024,Week-11-2024,2026-06-11T13:55:45.637Z,dbfs:/Volumes/ecommerce/raw/raw_landing/date/date.csv
20240326,2024-03-26,2024,March,Tuesday,0,Q1-2024,Week-13-2024,2026-06-11T13:55:45.637Z,dbfs:/Volumes/ecommerce/raw/raw_landing/date/date.csv


In [0]:
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.gold.gld_dim_date")